<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 16 → Projeto Integrador e MLOps
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Wedge
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
def finalizar(titulo, xlabel, ylabel):
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


# 1. TREINO
np.random.seed(42)
n = 200

X = pd.DataFrame({
    "temperatura": np.random.normal(65, 8, n),
    "vibracao": np.random.normal(2.8, 0.8, n)
})

y = ((X["temperatura"] > 72) & (X["vibracao"] > 3.2)).astype(int)

modelo = LogisticRegression().fit(X, y)
acuracia = accuracy_score(y, modelo.predict(X))

# 2. PRODUÇÃO
n = 60

df = pd.DataFrame({
    "tempo": pd.date_range("2026-10-05 08:00", periods=n, freq="5min"),
    "temperatura": np.random.normal(66, 7, n),
    "vibracao": np.random.normal(2.9, 0.7, n)
})

df.loc[40:47, "temperatura"] += np.linspace(5, 15, 8)
df.loc[40:47, "vibracao"] += np.linspace(0.5, 1.8, 8)

df["prob_falha"] = modelo.predict_proba(df[["temperatura", "vibracao"]])[:, 1]
df["status"] = np.where(df["prob_falha"] >= 0.5, "Manutenção", "Normal")

# 3. DADOS DE TREINO
plt.figure(figsize=(8, 5))
plt.scatter(X.loc[y == 0, "vibracao"], X.loc[y == 0, "temperatura"], label="Normal", alpha=0.7)
plt.scatter(X.loc[y == 1, "vibracao"], X.loc[y == 1, "temperatura"], label="Falha", alpha=0.8)
finalizar("1. Análise dos Dados de Treino", "Vibração", "Temperatura (°C)")

# 4. PRODUÇÃO
alertas = df[df["status"] == "Manutenção"]

plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["prob_falha"], linewidth=2, label="Probabilidade de falha")
plt.axhline(0.5, linestyle="--", linewidth=2, label="Threshold = 0.50")
plt.scatter(alertas["tempo"], alertas["prob_falha"], s=70, label="Alerta")
plt.ylim(0, 1.05)
finalizar("2. Modelo em Produção - Risco de Falha", "Tempo", "Probabilidade")

# 5. RESULTADO
contagem = df["status"].value_counts().reindex(["Normal", "Manutenção"], fill_value=0)

plt.figure(figsize=(6, 4))
barras = plt.bar(contagem.index, contagem.values)
plt.bar_label(barras, padding=3)
plt.title("3. Resultado Operacional em Produção")
plt.ylabel("Quantidade de leituras")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 6. RESULTADOS
print(f"Acurácia no treino: {acuracia:.1%}")
print("Leituras em produção:", len(df))
print("Alertas de manutenção:", int((df["status"] == "Manutenção").sum()))

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
def criar_dados(n, p_media, p_std, c_media, c_std, p_lim, c_lim):
    pressao = np.random.normal(p_media, p_std, n)
    corrente = np.random.normal(c_media, c_std, n)

    X = pd.DataFrame({
        "pressao_bar": pressao,
        "corrente_a": corrente
    })

    y = ((pressao < p_lim) & (corrente > c_lim)).astype(int)
    return X, y


def treinar(X, y):
    modelo = LogisticRegression().fit(X, y)
    acc = accuracy_score(y, modelo.predict(X))
    return modelo, acc


def histograma(treino, producao, coluna, titulo, xlabel):
    plt.figure(figsize=(8, 4))
    plt.hist(treino[coluna], bins=20, alpha=0.6, label="Treino")
    plt.hist(producao[coluna], bins=20, alpha=0.6, label="Produção")
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel("Frequência")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


np.random.seed(42)

# 1. TREINO
X_treino, y_treino = criar_dados(500, 6.0, 0.8, 12.0, 1.5, 5.2, 13.5)
modelo_v1, acc_treino = treinar(X_treino, y_treino)

# 2. PRODUÇÃO
X_prod, y_prod = criar_dados(300, 5.5, 1.0, 13.0, 1.8, 5.0, 14.0)

acc_prod_v1 = accuracy_score(y_prod, modelo_v1.predict(X_prod))
modelo_v2, acc_prod_v2 = treinar(X_prod, y_prod)

# 3. DRIFT
histograma(
    X_treino, X_prod, "pressao_bar",
    "Drift de Dados - Pressão Hidráulica", "Pressão (bar)"
)

histograma(
    X_treino, X_prod, "corrente_a",
    "Drift de Dados - Corrente Elétrica", "Corrente (A)"
)

# 4. DESEMPENHO
nomes = ["Modelo v1\nTreino", "Modelo v1\nProdução", "Modelo v2\nApós retreino"]
acuracias = [acc_treino*100, acc_prod_v1*100, acc_prod_v2*100]

plt.figure(figsize=(8, 4))
barras = plt.bar(nomes, acuracias)
plt.bar_label(barras, labels=[f"{v:.1f}%" for v in acuracias], padding=3)
plt.title("Desempenho do Modelo no Ciclo MLOps")
plt.ylabel("Acurácia (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)
plt.show()

# 5. MONITORAMENTO
periodos = ["Semana 1", "Semana 2", "Semana 3", "Semana 4", "Após retreino"]
desempenho = [96, 94, 90, 85, round(acc_prod_v2*100, 1)]

plt.figure(figsize=(9, 4))
plt.plot(periodos, desempenho, marker="o", linewidth=2)
plt.axhline(90, linestyle="--", linewidth=2, label="Meta mínima = 90%")
plt.title("Monitoramento Contínuo do Modelo")
plt.ylabel("Acurácia (%)")
plt.ylim(75, 100)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 6. RESULTADOS
print(f"Acurácia modelo v1 no treino: {acc_treino:.1%}")
print(f"Acurácia modelo v1 em produção: {acc_prod_v1:.1%}")
print(f"Acurácia modelo v2 após retreino: {acc_prod_v2:.1%}")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
def treinar(X, y):
    modelo = LogisticRegression(max_iter=1000).fit(X, y)
    return modelo, accuracy_score(y, modelo.predict(X))


def histograma(treino, prod, coluna, titulo, xlabel, cores):
    plt.figure(figsize=(9, 4))
    plt.hist(treino[coluna], bins=20, alpha=0.55, color=cores[0], label="Treino")
    plt.hist(prod[coluna], bins=20, alpha=0.55, color=cores[1], label="Produção")
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel("Frequência")
    plt.legend(title="Camadas")
    plt.grid(alpha=0.3)
    plt.show()


np.random.seed(42)
n = 600

# 1. TREINO
pressao = np.random.normal(6.2, 0.5, n)
vazao = np.random.normal(120, 8, n)
corrente = np.random.normal(11.5, 1.0, n)

X_treino = pd.DataFrame({
    "pressao": pressao,
    "vazao": vazao,
    "corrente": corrente
})

y_treino = ((pressao < 5.7) & (vazao < 114) & (corrente > 12.2)).astype(int)
modelo_v1, acc_treino = treinar(X_treino, y_treino)

# 2. PRODUÇÃO COM DRIFT
pressao_prod = np.random.normal(5.5, 0.7, n)
vazao_prod = np.random.normal(108, 11, n)
corrente_prod = np.random.normal(13.0, 1.3, n)

X_prod = pd.DataFrame({
    "pressao": pressao_prod,
    "vazao": vazao_prod,
    "corrente": corrente_prod
})

y_prod = ((pressao_prod < 5.2) | ((vazao_prod < 105) & (corrente_prod > 13.5))).astype(int)

acc_prod = accuracy_score(y_prod, modelo_v1.predict(X_prod))
modelo_v2, acc_v2 = treinar(X_prod, y_prod)

# 3. DRIFT
histograma(
    X_treino, X_prod, "pressao",
    "Data Drift - Pressão Hidráulica",
    "Pressão (bar)",
    ["royalblue", "tomato"]
)

histograma(
    X_treino, X_prod, "vazao",
    "Data Drift - Vazão",
    "Vazão (L/min)",
    ["seagreen", "darkorange"]
)

# 4. DEGRADAÇÃO
periodos = ["Treino", "Mês 1", "Mês 2", "Mês 3", "Mês 4", "Após retreino"]
desempenho = [
    round(acc_treino*100, 1), 95, 92, 88,
    round(acc_prod*100, 1), round(acc_v2*100, 1)
]

plt.figure(figsize=(9, 4))
plt.plot(periodos, desempenho, marker="o", linewidth=2, color="purple")
plt.axhline(90, color="red", linestyle="--", linewidth=2, label="Meta mínima = 90%")
plt.title("Degradação do Modelo em Produção")
plt.ylabel("Acurácia (%)")
plt.ylim(70, 100)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 5. ANTES E DEPOIS
nomes = ["Treino", "Produção\ncom drift", "Após\nretreino"]
valores = [acc_treino*100, acc_prod*100, acc_v2*100]

plt.figure(figsize=(7, 4))
barras = plt.bar(nomes, valores, color=["seagreen", "tomato", "royalblue"])
plt.bar_label(barras, labels=[f"{v:.1f}%" for v in valores], padding=3)
plt.title("Impacto do Drift e do Retreino")
plt.ylabel("Acurácia (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)
plt.show()

# 6. RESULTADOS
print(f"Acurácia no treino: {acc_treino:.1%}")
print(f"Acurácia em produção: {acc_prod:.1%}")
print(f"Acurácia após retreino: {acc_v2:.1%}")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
def rotulos(barras, valores, sufixo=""):
    plt.bar_label(barras, labels=[f"{v}{sufixo}" for v in valores], padding=3)


np.random.seed(42)

# 1. DADOS
df = pd.DataFrame({
    "dispositivo": [
        "Acesso", "Câmeras", "Iluminação", "Ocupação",
        "Estacionamento", "Climatização", "Elevadores", "Energia"
    ],
    "tentativas_falhas": np.random.randint(1, 18, 8),
    "dados_antes": np.random.randint(70, 120, 8),
    "dados_depois": np.random.randint(35, 75, 8)
})

kpis = pd.DataFrame({
    "Pilar": ["Governança", "Segurança", "Ética"],
    "Conformidade": [92, 84, 88]
})

# 2. CONFORMIDADE
plt.figure(figsize=(7, 4))
barras = plt.bar(kpis["Pilar"], kpis["Conformidade"],
                 color=["royalblue", "tomato", "seagreen"])
plt.axhline(90, color="black", linestyle="--", label="Meta = 90%")
rotulos(barras, kpis["Conformidade"], "%")
plt.title("Conformidade dos Pilares do Projeto IoT")
plt.ylabel("Conformidade (%)")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. SEGURANÇA
cores = [
    "tomato" if v >= 12 else "darkorange" if v >= 7 else "steelblue"
    for v in df["tentativas_falhas"]
]

plt.figure(figsize=(9, 4))
barras = plt.bar(df["dispositivo"], df["tentativas_falhas"], color=cores)
rotulos(barras, df["tentativas_falhas"])
plt.title("Tentativas de Acesso Falhas")
plt.ylabel("Quantidade")
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.3)
plt.show()

# 4. ÉTICA
x = np.arange(len(df))
largura = 0.36

plt.figure(figsize=(10, 4))
plt.bar(x - largura/2, df["dados_antes"], largura,
        color="mediumpurple", label="Antes")
plt.bar(x + largura/2, df["dados_depois"], largura,
        color="seagreen", label="Após minimização")
plt.title("Redução da Coleta de Dados")
plt.ylabel("Volume relativo de dados")
plt.xticks(x, df["dispositivo"], rotation=25)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 5. RESULTADOS
reducao = (1 - df["dados_depois"].sum() / df["dados_antes"].sum()) * 100

print("Conformidade média:", round(kpis["Conformidade"].mean(), 1), "%")
print("Tentativas falhas:", int(df["tentativas_falhas"].sum()))
print("Redução de dados coletados:", round(reducao, 1), "%")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

In [ ]:
def rotulos(ax, barras, valores, formato="{}"):
    ax.bar_label(barras, labels=[formato.format(v) for v in valores], padding=3)


# 1. DADOS
df = pd.DataFrame({
    "setor": ["Setor A", "Setor B", "Setor C", "Setor D", "Setor E"],
    "vazao_l_min": [118, 121, 95, 119, 116],
    "pressao_bar": [4.8, 4.7, 3.1, 4.6, 4.9],
    "perda_pct": [4, 5, 18, 6, 3],
    "alertas": [1, 2, 8, 2, 1]
})

df["score_risco"] = (
    df["perda_pct"]*2 + df["alertas"]*3
    + np.maximum(4 - df["pressao_bar"], 0)*10
)

monitorados = len(df)
alertas = int(df["alertas"].sum())
criticos = int((df["perda_pct"] >= 10).sum())
prioridade = df.loc[df["score_risco"].idxmax(), "setor"]

# 2. DASHBOARD
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(3, 12, height_ratios=[1.2, 4, 4], hspace=0.6, wspace=0.8)

fig.suptitle(
    "Dashboard - Comunicação dos Resultados do Projeto IoT",
    fontsize=18, fontweight="bold", y=0.98
)

# 3. CARDS
cards = [
    ("Setores Monitorados", monitorados, "#4C78A8"),
    ("Total de Alertas", alertas, "#F58518"),
    ("Setores Críticos", criticos, "#E45756"),
    ("Prioridade", prioridade, "#54A24B")
]

for i, (titulo, valor, cor) in enumerate(cards):
    ax = fig.add_subplot(gs[0, i*3:(i+1)*3])
    ax.axis("off")
    ax.text(0.5, 0.65, titulo, ha="center", fontsize=12, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor=cor, alpha=0.20, edgecolor=cor))
    ax.text(0.5, 0.25, str(valor), ha="center", fontsize=18, fontweight="bold", color=cor)

# 4. PERDA DE ÁGUA
ax1 = fig.add_subplot(gs[1, :6])
cores = ["seagreen" if v < 10 else "tomato" for v in df["perda_pct"]]
b = ax1.bar(df["setor"], df["perda_pct"], color=cores)
ax1.axhline(10, color="orange", linestyle="--", linewidth=2, label="Limite = 10%")
ax1.set(title="Perda de Água por Setor", ylabel="Perda (%)")
ax1.title.set_fontweight("bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)
rotulos(ax1, b, df["perda_pct"], "{}%")

# 5. PRIORIDADE
ax2 = fig.add_subplot(gs[1, 6:])
ordem = df.sort_values("score_risco", ascending=False)
b = ax2.bar(ordem["setor"], ordem["score_risco"],
            color=["red", "orange", "royalblue", "seagreen", "mediumpurple"])
ax2.set(title="Prioridade de Ação", ylabel="Score de risco")
ax2.title.set_fontweight("bold")
ax2.grid(axis="y", alpha=0.3)
rotulos(ax2, b, ordem["score_risco"], "{:.1f}")

# 6. RESUMO
ax3 = fig.add_subplot(gs[2, :6])
valores = [monitorados, alertas, criticos]
b = ax3.bar(
    ["Setores\nmonitorados", "Alertas", "Setores\ncríticos"],
    valores,
    color=["steelblue", "darkorange", "tomato"]
)
ax3.set(title="Resumo Executivo", ylabel="Quantidade")
ax3.title.set_fontweight("bold")
ax3.grid(axis="y", alpha=0.3)
rotulos(ax3, b, valores)

# 7. TABELA
ax4 = fig.add_subplot(gs[2, 6:])
ax4.axis("off")

tabela = df.copy()
tabela["pressao_bar"] = tabela["pressao_bar"].round(1)
tabela["score_risco"] = tabela["score_risco"].round(1)

tbl = ax4.table(
    cellText=tabela.values,
    colLabels=tabela.columns,
    loc="center",
    cellLoc="center"
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.1, 1.7)

for (linha, _), cell in tbl.get_celld().items():
    cell.set_facecolor("#4C78A8" if linha == 0 else "#F7F7F7")
    if linha == 0:
        cell.set_text_props(weight="bold", color="white")

ax4.set_title("Tabela Resumo dos Setores", fontweight="bold", pad=12)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()